# FLIGHT: stitching and variational-rank ablations

Enable Internet and a GPU in Kaggle. Attach gpw-kaggle-inputs containing
airline.npz and airline.json at INPUT_ROOT. Both files are required;
checksums/provenance are verified. No pickle reader or substitute source.

Six runs: FLIGHT only, model seed31, ranks0/64 crossed with no, fixed and
learned stitches. Use1024 inducing points in8 patches of128. All six
runs share identical initial observed rows, supports, centers, candidate
graph and candidate stitches, checked by a saved geometry fingerprint.
Separate local ARD RBF kernels and variational parameters remain free.
None removes constraints, fixed freezes only stitches, learned moves them.
Preserve prior-dependent q initialization; initial q covariance can differ.

Uniform800,000-row subset without replacement from the5,929,413-row
trusted author-prepared FLIGHT file, eight features,subset seed31 and
80/20 split with data seed31:640,000 train/160,000 test. Input min-max
scaling uses all selected rows; target scaling uses training rows only.
This is size-matched, not the paper's exact records or90/10 split.
Source-row indices and checksums are saved in every run archive.
Existing FLIGHT batch size,
learning rates, initialization, bounds, numerics and stitch count remain.
The1024-support ablation budget differs from the2048-point main preset.

Defaults100 epochs maximum/20 kernel-noise warm-up; training-loss early
stopping patience10, relative improvement0.001, evaluate the last state.
Supports and learned stitches are not frozen during warm-up. Logs name
dataset/model/seed/rank/stitch mode every20 epochs plus initial/final state;
histories save every epoch. Change EPOCHS=3,WARMUP_EPOCHS=1 for an explicit
short smoke check. No configuration is adjusted automatically.

Review the complete plan, then set RUN_TRAINING=True. Code loads latest
main in fresh workers without reconnects. Every completed model saves
predictions, metrics, initial/final positions, histories, checkpoints,
effective configuration/provenance and a ZIP before the next begins.
Failures stop subsequent runs. Reruns start fresh, never resume/skip.
Download ZIPs or save Kaggle outputs: session-local files are not a backup.


## 1. Load latest source and dependencies


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['checkout', 'install', 'settings', 'preflight', 'data', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in []:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
import hashlib, importlib, json, os, subprocess, sys
from pathlib import Path

REPOSITORY = 'https://github.com/chahineNejm/Generalized-Patchwork.git'
BRANCH = 'main'
ROOT = Path('/kaggle/working/gpw_source')
os.environ["GIT_TERMINAL_PROMPT"] = "0"
if ROOT.exists():
    top = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "--show-toplevel"], text=True).strip()
    if Path(top).resolve() != ROOT.resolve():
        raise ValueError("Existing checkout must be its own Git repository")
    remote = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
    branch = subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip()
    if remote != REPOSITORY or branch != BRANCH:
        raise ValueError("Existing checkout repository or branch does not match the requested source")
    subprocess.run(["git", "-C", str(ROOT), "diff", "--exit-code", "HEAD"], check=True)
    untracked = subprocess.check_output(["git", "-C", str(ROOT), "ls-files", "--others",
        "--exclude-standard", "--", ":(glob)src/**/*.py", ":(glob)experiments/**/*.py"], text=True).strip()
    if untracked:
        raise ValueError(f"Untracked source files prevent verified reuse: {untracked}")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(ROOT), "merge-base", "--is-ancestor", "HEAD", "origin/" + BRANCH], check=True)
    subprocess.run(["git", "-C", str(ROOT), "merge", "--ff-only", "origin/" + BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH,
                    "--single-branch", REPOSITORY, str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "sparse-checkout", "set", "--no-cone",
                "/src/", "/experiments/", "/pyproject.toml", "/README.md"], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", BRANCH], check=True)

COMMIT = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
SOURCE_ZIP = subprocess.check_output(["git", "-C", str(ROOT), "archive", "--format=zip",
                                      COMMIT, "pyproject.toml", "README.md", "src", "experiments"])
SOURCE_PROVENANCE = {"repository": REPOSITORY, "branch": BRANCH, "commit": COMMIT,
                     "source_sha256": hashlib.sha256(SOURCE_ZIP).hexdigest()}
os.environ["MPLCONFIGDIR"] = str(ROOT / "results/.matplotlib")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.chdir(ROOT)
for path in reversed((str(ROOT / "src"), str(ROOT))):
    if path not in sys.path:
        sys.path.insert(0, path)
importlib.invalidate_caches()
_GPW_CELL_RESULT = print(SOURCE_PROVENANCE)
_OISST_STAGES['checkout'] = True
_GPW_CELL_RESULT


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['install', 'settings', 'preflight', 'data', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['checkout']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "--editable", str(ROOT) + "[data,ukhousing,benchmark]"], check=True)
import runpy
run_worker = runpy.run_path(str(ROOT / "experiments/colab_client.py"))["run_worker"]
WORK_ROOT = Path("/kaggle/working/gpw_ablation_results")
DATA_ROOT = Path("/kaggle/working/gpw_data")

def call_worker(action, *, on_event=None, **values):
    return run_worker(ROOT, {"action": action, "source_provenance": SOURCE_PROVENANCE, **values},
        module="experiments.multidataset_worker", jobs_directory=WORK_ROOT / "worker_jobs", on_event=on_event)
_OISST_STAGES['install'] = True


## 2. Review the six-run FLIGHT plan

DATASETS=["flight"], SEEDS=[31], RANKS=[0,64] and all three stitch modes.
DATA_OVERRIDES explicitly selects800,000 rows with subset_size. INPUT_ROOT
must contain both airline.npz and airline.json. Hyperparameters
remain editable here; MODEL_OVERRIDES applies equally to all six variants.
Data preprocessing and seed remain unchanged. One seed cannot estimate
between-seed variability; summary sample SD is undefined, not zero.


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['settings', 'preflight', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['install']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
from datetime import UTC, datetime
from IPython.display import FileLink, Image, Markdown, display
import pandas as pd

INPUT_ROOT = "/kaggle/input/datasets/chahinenejma/gpw-kaggle-inputs"
DATASETS = ['flight']
SEEDS = [31]
RANKS = [0, 64]
STITCH_MODES = ['none', 'fixed', 'learned']
INDUCING_POINTS = 1024
PATCHES = 8
EPOCHS = 100
WARMUP_EPOCHS = 20
EARLY_STOPPING = {'patience': 10, 'relative_min_improvement': 0.001}
MODEL_OVERRIDES = {}
DATA_OVERRIDES = {'flight': {'subset_size': 800000}}
SAVE_INDUCING_POSITIONS = True
PROGRESS_EVERY_EPOCHS = 20

settings = call_worker("settings", study="gpw_stitch_rank", device="cuda:0", input_root=INPUT_ROOT,
    datasets=DATASETS, seeds=SEEDS, ranks=RANKS, stitch_modes=STITCH_MODES,
    inducing_points=INDUCING_POINTS, patches=PATCHES, epochs=EPOCHS, warmup_epochs=WARMUP_EPOCHS,
    early_stopping=EARLY_STOPPING, model_overrides=MODEL_OVERRIDES, data_overrides=DATA_OVERRIDES,
    progress_every_epochs=PROGRESS_EVERY_EPOCHS,
    save_inducing_positions=SAVE_INDUCING_POSITIONS)
PLAN, PLAN_SHA256 = settings["plan"], settings["plan_sha256"]
display(pd.DataFrame(settings["summary"]))
print("Selected runs:", settings["total_runs"], "Maximum total epochs:", settings["total_epochs"])
print("Study:", PLAN["study"])
_GPW_CELL_RESULT = display(pd.DataFrame([{"dataset": r["dataset"], "name": r["name"], **r["config"]} for r in PLAN["runs"]]))
_OISST_STAGES['settings'] = True
_GPW_CELL_RESULT


## 3. Check inputs, requested GPU and available disk


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['preflight', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['install', 'settings']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
PREFLIGHT_DIRECTORY = WORK_ROOT / datetime.now(UTC).strftime("preflight_%Y%m%dT%H%M%S%fZ")
_GPW_CELL_RESULT = print(call_worker("preflight", plan=PLAN, plan_sha256=PLAN_SHA256, directory=str(PREFLIGHT_DIRECTORY)))
_OISST_STAGES['preflight'] = True
_GPW_CELL_RESULT


## 4. Prepare once, or explicitly load verified prepared arrays

Keep this stage untouched when changing only the training selection.
To reuse arrays, set PREPARED_ACTION="load" and PREPARED_PATH to the exact
prepared folder, including one from the previous multi-dataset notebook
if configurations/checksums match. Loading never rebuilds failed data.


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['data', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['install', 'settings', 'preflight']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
PREPARED_ACTION = "prepare"
PREPARED_PATH = None
if PREPARED_ACTION == "prepare":
    PREPARED_DIRECTORY = WORK_ROOT / datetime.now(UTC).strftime("prepared_%Y%m%dT%H%M%S%fZ")
    prepared = call_worker("prepare", plan=PLAN, plan_sha256=PLAN_SHA256,
        directory=str(PREPARED_DIRECTORY), data_root=str(DATA_ROOT))
elif PREPARED_ACTION == "load":
    if PREPARED_PATH is None:
        raise ValueError("Set PREPARED_PATH explicitly for verified data reuse")
    PREPARED_DIRECTORY = Path(PREPARED_PATH)
    prepared = call_worker("load", plan=PLAN, plan_sha256=PLAN_SHA256, directory=str(PREPARED_DIRECTORY))
else:
    raise ValueError("PREPARED_ACTION must be prepare or load")
_GPW_CELL_RESULT = print(prepared)
_OISST_STAGES['data'] = True
_GPW_CELL_RESULT


## 5. Authorize this batch and train

Set RUN_TRAINING=True after reviewing the complete plan. Training history
updates one display rather than appending a figure every snapshot. Final
plots are saved on disk; completed-run links appear immediately. Failure
or interruption stops the worker. Rerun settings/preflight after changes;
this cell never automatically skips or resumes earlier models.


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['install', 'settings', 'preflight', 'data']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
RUN_TRAINING = False
LIVE_PLOTS = True
if RUN_TRAINING is not True:
    raise RuntimeError("Training disabled. Review the plan, then explicitly set RUN_TRAINING=True")
training_view = display("Starting GPW ablations...", display_id=True)
metrics_view = display("No completed run yet.", display_id=True)

def show_event(event):
    path = Path(event["directory"])
    if event["kind"] == "snapshot":
        if LIVE_PLOTS:
            training_view.update(Image(filename=str(path / "visuals/training.png")))
    elif event["kind"] == "complete":
        metrics_view.update(pd.DataFrame([{"dataset": event["dataset"], "variant": event["name"], **event["metrics"]}]))
        print(event["dataset"], "/", event["name"], "saved:", event["archive"])
        display(FileLink(event["archive"]))
    else:
        raise ValueError(f"Unknown worker event: {event['kind']}")

BATCH_DIRECTORY = WORK_ROOT / datetime.now(UTC).strftime("ablations_%Y%m%dT%H%M%S%fZ")
result = call_worker("train", plan=PLAN, plan_sha256=PLAN_SHA256,
    prepared_directory=str(PREPARED_DIRECTORY), directory=str(BATCH_DIRECTORY),
    execute_training=RUN_TRAINING, on_event=show_event)
display(pd.read_csv(BATCH_DIRECTORY / "comparison.csv"))
aggregate = json.loads((BATCH_DIRECTORY / "ablation_summary.json").read_text())
display(pd.DataFrame([{"dataset": r["dataset"], "rank": r["q_rank"], "stitches": r["stitch_mode"],
    "completed_seeds": r["completed_seeds"], **{k + "_mean": v["mean"] for k, v in r["metrics"].items()}}
    for r in aggregate]))
for dataset in DATASETS:
    display(Markdown("### " + dataset))
    display(Image(filename=str(BATCH_DIRECTORY / "visuals" / (dataset + ".png"))))
_GPW_CELL_RESULT = print("Full seed means and sample SDs:", BATCH_DIRECTORY / "ablation_summary.json")
_OISST_STAGES['training'] = True
_GPW_CELL_RESULT


## 6. Archive the completed batch


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['archive']:
    _OISST_STAGES[_stage] = False
for _required in ['training']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
ARCHIVE_PATH = WORK_ROOT / datetime.now(UTC).strftime("gpw_ablations_%Y%m%dT%H%M%S%fZ.zip")
archived = call_worker("archive", plan=PLAN, plan_sha256=PLAN_SHA256,
    directory=str(BATCH_DIRECTORY), destination=str(ARCHIVE_PATH))
display(FileLink(archived["archive"]))
_GPW_CELL_RESULT = print("Save a Kaggle version with outputs or download the archives to preserve them outside this session.")
_OISST_STAGES['archive'] = True
_GPW_CELL_RESULT
